# 2026-IoT 코랩 실행 노트북

위에서부터 차례로 실행한다. 런타임이 끊기면 1번부터 다시 실행한다. 5번 실험 셀은 끝난 실행을 건너뛰고, 중간에 끊긴 실행은 내 드라이브(`WORK`)의 체크포인트부터 이어 돈다.

**처음 한 번만 할 일**
1. 메뉴 `런타임 > 런타임 유형 변경`에서 GPU(T4)를 고른다.
2. 왼쪽 열쇠 아이콘(Secrets)에 `GH_TOKEN`을 추가하고 이 노트북의 액세스를 켠다.
   - 값은 `combile/2026-IoT` 읽기 권한이 있는 GitHub 토큰이다. 저장소가 비공개라 토큰이 필요하다.

## 0. 설정

In [ ]:
BRANCH = 'main'                  # 아직 합치지 않은 브랜치를 시험할 때는 'feature/defense' 등으로 바꾼다
RESULTS = '/content/drive/MyDrive/2026-IoT/results'   # 공유 결과 폴더. 끝난 실행의 결과만 한 번 복사한다
WORK = '/content/drive/MyDrive/2026-IoT-work'   # 내 드라이브의 작업 폴더. 실행 중인 결과·체크포인트를 둔다
# 공유 폴더(바로가기)에 있는 파일을 덮어쓰거나 지우면 드라이브에 같은 이름 파일이 계속 새로 생긴다.
# 그래서 라운드마다 고쳐 쓰는 파일은 내 드라이브(WORK)에 쓰고, 공유 폴더에는 새 파일을 한 번만 만든다.
DATA = '/content/drive/MyDrive/2026-IoT/data'   # CIFAR-10을 드라이브에 한 번만 받아 둔다(런타임이 끊겨도 다시 받지 않음)

## 1. GPU 확인, 드라이브 연결, 코드 받기

In [ ]:
import os; os.chdir('/content')   # 2번 셀이 옮긴 작업 폴더(/content/FedACG)를 지우기 전에 빠져나온다
!nvidia-smi -L
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, subprocess, shutil, pathlib
os.makedirs(RESULTS, exist_ok=True)
os.makedirs(WORK, exist_ok=True)
token = userdata.get('GH_TOKEN')
shutil.rmtree('/content/FedACG', ignore_errors=True)
shutil.rmtree('/content/2026-IoT', ignore_errors=True)
subprocess.run(['git', 'clone', '-q', 'https://github.com/geehokim/FedACG.git', '/content/FedACG'], check=True)
subprocess.run(['git', 'clone', '-q', '-b', BRANCH, f'https://{token}@github.com/combile/2026-IoT.git',
                '/content/2026-IoT'], check=True)   # 토큰이 화면에 찍히지 않게 subprocess로 실행한다
print(subprocess.run(['git', '-C', '/content/2026-IoT', 'log', '--oneline', '-3'],
                     capture_output=True, text=True).stdout)

## 2. 팀 코드 붙이기

`attacks/`, `metrics/`, `defenses/`, `resume/`와 설정 폴더를 FedACG에 복사하고 패치를 순서대로 적용한다. 아직 저장소에 없는 모듈은 건너뛴다.

`resume_integration.patch`는 끊긴 실행을 이어 돌리게 한다. 체크포인트는 작업 폴더(`WORK`, 내 드라이브) 아래 `checkpoints/`에 저장된다.

`colab_memory.patch`는 코랩 무료판 메모리(12.7GB) 때문에 넣는다. FedACG 원본은 클라이언트 100명분 모델을 처음에 모두 만들어 메모리 부족으로 멈춘다. 이 패치는 클라이언트 객체 하나를 돌려 쓰고, 결과는 원본과 같다.

In [ ]:
T = pathlib.Path('/content/2026-IoT')
os.chdir('/content/FedACG')
for pkg in ['attacks', 'metrics', 'defenses', 'resume']:
    if (T / pkg).exists():
        shutil.copytree(T / pkg, pkg, dirs_exist_ok=True)
for grp in (T / 'configs').iterdir():
    if grp.is_dir() and grp.name != 'experiments':
        shutil.copytree(grp, f'configs/{grp.name}', dirs_exist_ok=True)
for p in ['fedacg_integration.patch', 'metrics_integration.patch', 'defense_integration.patch', 'colab_memory.patch',
          'resume_integration.patch']:
    if (T / 'integration' / p).exists():
        subprocess.run(['git', 'apply', str(T / 'integration' / p)], check=True)
        print('applied', p)
HAS_METRICS = (T / 'metrics').exists()
print('metrics module:', HAS_METRICS)

## 3. 패키지 설치와 테스트

In [ ]:
!pip -q install hydra-core==1.3.2 hydra-colorlog omegaconf imageio coloredlogs tabulate torchinfo torchmetrics wandb pytest
!cd /content/2026-IoT && python -m pytest -q

## 4. 2라운드 점검

모델 오염 30% + Krum으로 2라운드만 돌린다. 오류 없이 끝나는지, 라운드당 몇 초 걸리는지 확인한다. 이 시간으로 전체 실험 시간을 계산한다(라운드당 시간 × 라운드 수 × 실행 횟수).

In [ ]:
COMMON = f'dataset.path={DATA} dataset.client_path=/content/FedACG/client_data wandb=False'
if HAS_METRICS:
    COMMON += f' metrics.results_dir={WORK}'   # 점검 결과는 공유 폴더에 올리지 않는다
import time
t0 = time.time()
!python federated_train.py server=FedACG client=ACG dataset=cifar10 attack=mp attack.ratio=0.3 attack.start_round=0 attack.end_round=2 +defense=default +server.aggregator=krum trainer.global_rounds=2 trainer.num_clients=100 trainer.participation_rate=0.1 exp_name=smoke {COMMON}
print(f'2라운드 {time.time() - t0:.0f}초 (데이터 다운로드 시간 포함)')

## 5. 실험 실행 (사람마다 `WORKER`, `SPEC`, `PART`만 바꾼다)

| 사람 | `SPEC` | 실행 수 |
|---|---|---|
| 예다은 | `lf_sweep.yaml` (라벨 플리핑) | 90 |
| 우은식 | `mp_sweep.yaml` (모델 오염) | 90 |
| 강윤서 | `baseline_sweep.yaml` (무공격 기준선) | 30 |

- 결과는 공유 드라이브 폴더 `MyDrive/2026-IoT/results`에 모인다. 처음에는 공유받은 `2026-IoT` 폴더를 **내 드라이브에 바로가기로 추가**한다.
  - 실행 중에는 결과 CSV·공격 기록·로그·체크포인트를 내 드라이브의 `WORK`(`MyDrive/2026-IoT-work`)에 쓴다. 실행이 끝나면 결과 CSV·공격 기록·로그를 공유 `results/`로 한 번 복사한다.
  - 바로가기로 연결한 남의 폴더에서 파일을 덮어쓰거나 지우면 같은 이름 파일이 계속 새로 생기기 때문이다(폴더 주인은 해당 없음).
- 실행 순서는 우선순위 순이다: 시드 1의 `priority_ratio`(20%) 조합 → 시드 1의 나머지 비율 → 시드 2 → 시드 3. 일찍 멈춰도 핵심 결과부터 남는다.
- **같은 `SPEC`을 여러 명이 나눠 돌릴 때**: `PART`에 `(내 번호, 인원)`을 넣는다. 예: 2명이면 한 명은 `(0, 2)`, 다른 한 명은 `(1, 2)`. 혼자 돌리면 `None`이다.
  - 내 몫을 다 끝내고 남의 몫을 도와줄 때는 `PART = None`으로 돌린다.
  - 다른 사람이 돌리는 중인 실행(`results/locks/`의 잠금 파일)은 건너뛴다.
- 결과 CSV의 마지막 라운드가 `ROUNDS - 1`이면 끝난 실행으로 보고 건너뛴다.
- 런타임이 끊기면 1~3번을 다시 실행한 뒤 이 셀을 실행한다(4번 점검은 건너뛰어도 된다).
  - 중간에 끊긴 실행은 `WORK/checkpoints/<결과 이름>.pt`부터 이어 돈다(`resume.every`라운드마다 저장, 기본 10).
  - 끝난 실행의 체크포인트는 지운다.
  - `WORK`에서는 끝났는데 공유 폴더로 복사하기 전에 끊긴 실행은 다시 돌리지 않고 복사만 한다.
- `ROUNDS`와 `EXTRA`(공격 구간 등)는 팀 전체가 같은 값을 써야 한다. 바꾸기 전에 팀에 알린다. 현재 값은 README의 1차 실험 계획(80라운드, 공격 30~49)이다.

In [ ]:
# ---- 사람마다 바꾸는 값 ----
WORKER = '예다은'      # 잠금 파일에 남는 이름. 사람마다 다르게 쓴다
SPEC = '/content/2026-IoT/configs/experiments/lf_sweep.yaml'   # 우은식: mp_sweep.yaml / 강윤서: baseline_sweep.yaml
PART = None           # 혼자면 None. 같은 SPEC을 2명이 나누면 (0, 2)와 (1, 2)

# ---- 팀 전체가 같은 값 ----
ROUNDS = 80           # 1차 계획 (README). 2차에서 늘리면 끝난 실행도 이어 돈다
EXTRA = 'attack.start_round=30 attack.end_round=50'   # 1차 계획의 공격 구간 (README)
STALE_MINUTES = 150   # 잠금을 건 뒤 이 시간이 지난 다른 사람의 잠금은 무시한다(런타임이 죽은 것으로 봄)
                      # 결과는 실행이 끝나야 공유 폴더에 올라오므로 80라운드 1회(약 65분)보다 넉넉히 잡는다

import sys, yaml
sys.path.insert(0, '/content/2026-IoT')
from scripts import run_queue as rq

# 4번 점검 셀을 건너뛰어도 되게 여기서도 정한다
COMMON = f'dataset.path={DATA} dataset.client_path=/content/FedACG/client_data wandb=False metrics.results_dir={WORK}'

def publish(name):
    """WORK에서 끝난 실행의 결과 CSV·공격 기록·로그를 공유 폴더로 한 번 복사한다."""
    stem = pathlib.Path(name).stem
    for rel in [name, f'attack_logs/{name}', f'logs/{stem}.log']:
        src = pathlib.Path(WORK) / rel
        if src.exists():
            dst = pathlib.Path(RESULTS) / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(src, dst)

spec = yaml.safe_load(open(SPEC))
mine = rq.my_share(rq.prioritized(spec), PART)
queue = rq.todo(spec, RESULTS, ROUNDS, PART)
print(f'{WORKER}: 내 몫 {len(mine)}개 중 {len(mine) - len(queue)}개 끝남, {len(queue)}개 남음')
if not HAS_METRICS:
    print('metrics 모듈이 없어 결과 CSV가 생기지 않는다. 2번 셀을 다시 확인한다.')
else:
    for i, (name, cmd) in enumerate(queue, 1):
        if rq.finished(RESULTS, name, ROUNDS):
            continue   # 기다리는 동안 다른 사람이 끝냄
        other = rq.held_by_other(RESULTS, name, WORKER, STALE_MINUTES)
        if other:
            print(f'[{i}/{len(queue)}] {name}: {other} 님이 돌리는 중이라 건너뜀')
            continue
        if rq.finished(WORK, name, ROUNDS):
            publish(name)   # 다 돌았는데 복사 전에 끊긴 실행
            print(f'[{i}/{len(queue)}] {name}: 이미 끝나서 공유 폴더로 복사만 함')
            continue
        rq.acquire(RESULTS, name, WORKER)
        print(f'[{i}/{len(queue)}] {name}')
        try:
            subprocess.run(f'{cmd} trainer.global_rounds={ROUNDS} {EXTRA} {COMMON}', shell=True, check=True)
            publish(name)
        finally:
            rq.release(RESULTS, name, WORKER)